# 1. Install and Import Dependencies

In [1]:
# For Apple Silicon (M1/M2) macOS
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu


In [146]:
!pip install transformers requests beautifulsoup4 pandas numpy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [2]:
import torch
print(torch.__version__)
print(torch.backends.mps.is_available())  # True if Metal GPU is available


2.8.0
True


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
import re

/opt/anaconda3/envs/THESIS/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Instantiate Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')


# tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")
# model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")


# 3. Encode and Calculate Sentiment

In [5]:
tokens = tokenizer.encode('It was good but couldve been better. Great', return_tensors='pt')

In [6]:
result = model(tokens)

In [7]:
result.logits

tensor([[-2.7768, -1.2353,  1.4419,  1.9804,  0.4584]],
       grad_fn=<AddmmBackward0>)

In [8]:
int(torch.argmax(result.logits))+1

4

# 4. Collect Reviews

In [9]:
import json
filename = "llm_check_reviews/split_1_no_name.json"
with open(filename, "r", encoding = "utf-8") as f:
    final_reviews = json.load(f)

In [10]:
# filename = "llm_check_reviews/mentioned_reviews_spacy_1.json"
# with open(filename, "r", encoding = "utf-8") as f:
#     final_reviews = json.load(f)

In [131]:
final_reviews[0:2]

[{'model_id': 'LoneStriker/Meta-Llama-3-8B-Instruct-GGUF',
  'topic': 'Meta-Llama-3-8B-Instruct-GGUF',
  'reddit': [{'id': None,
    'mentioned': ['```bash\npython ai_server.py openai --model "lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF"\n```\n*(You might need to adjust the `--api-base` if your server isn\'t at the default `http://localhost:1234/v1`)*\n\nYou can also connect to OpenAI and every service that is OpenAI compatible and use their models.\n',
     '# Example for OLLAMA:\npython ai_server.py ollama --model llama3\n\n# Example for OpenAI-compatible (e.g., LM Studio):\npython ai_server.py openai --model "lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF"\n"""\nimport http.server\nimport socketserver\nimport os\nimport argparse\nimport re\nfrom urllib.parse import urlparse, parse_qs\n\n# Conditionally import libraries\ntry:\n    import openai\nexcept ImportError:\n    openai = None\ntry:\n    import ollama\nexcept ImportError:\n    ollama = None\n\n# --- 1.',
     '```bash\n

# 5. Load Reviews into DataFrame

In [156]:
import numpy as np
import pandas as pd
from pandas import json_normalize
import re

In [157]:
import pandas as pd

rows = []

for model_m in final_reviews:
    model_id = model_m["model_id"]
    topic = model_m["topic"]
    
    # Iterate over all sources
    for source_name in ["reddit", "hf", "stackoverflow"]:
        for entry in model_m.get(source_name, []):
            score = entry.get("score")
            mentioned_texts = entry.get("mentioned", [])
            
            # Each mention gets its own row
            for text in mentioned_texts:
                rows.append({
                    "model_id": model_id,
                    "topic": topic,
                    "source": source_name,
                    "score": score,
                    "reviews": text
                })

# Create the DataFrame
df = pd.DataFrame(rows)

# Preview
print(df.head())

                                            model_id  \
0          LoneStriker/Meta-Llama-3-8B-Instruct-GGUF   
1  SandLogicTechnologies/Meta-Llama-3-8B-Instruct...   
2            AI-Engine/Meta-Llama-3-8B-Instruct-GGUF   
3             PawanKrd/Meta-Llama-3-8B-Instruct-GGUF   
4        MaziyarPanahi/Meta-Llama-3-8B-Instruct-GGUF   

                           topic  source  score  \
0  Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
1  Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
2  Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
3  Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
4  Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   

                                             reviews  
0  https://preview.redd.it/w18ytwq4naxc1.jpg?widt...  
1  https://preview.redd.it/w18ytwq4naxc1.jpg?widt...  
2  https://preview.redd.it/w18ytwq4naxc1.jpg?widt...  
3  https://preview.redd.it/w18ytwq4naxc1.jpg?widt...  
4  https://preview.redd.it/w18ytwq4naxc1.jpg?widt...  


In [158]:
df = df[df['topic'] != 'xyg']
# df.to_csv('llm_check_reviews/output.csv', index=False)

In [159]:
# Identify duplicates based on the 'mentioned' column
duplicates_mask = df.duplicated(subset=["reviews"], keep="first")

# DataFrame with unique reviews
df_unique = df[~duplicates_mask].reset_index(drop=True)

# DataFrame with duplicate reviews
df_duplicates = df[duplicates_mask].reset_index(drop=True)


In [160]:
print("Total:", len(df))
print("Unique:", len(df_unique))
print("Duplicate:", len(df_duplicates))

Total: 21686
Unique: 14632
Duplicate: 7054


In [ ]:
# # Save DataFrame to CSV
df_unique.to_csv("llm_check_reviews/mentioned_reviews_unique.csv", index=False, encoding="utf-8")
df_duplicates.to_csv("llm_check_reviews/mentioned_reviews_duplicates.csv", index=False, encoding="utf-8")
# df_unique.to_csv("llm_check_reviews/mentioned_reviews_unique_spacy.csv", index=False, encoding="utf-8")
# df_duplicates.to_csv("llm_check_reviews/mentioned_reviews_duplicates_spacy.csv", index=False, encoding="utf-8")

In [161]:
from nltk.tokenize import sent_tokenize
text = df_unique['reviews'][5]
sentences = sent_tokenize(text)
print(len(sentences))


1


# 6. Preprocessing

In [162]:
df_unique['reviews'].iloc[0]

'https://preview.redd.it/w18ytwq4naxc1.jpg?width=887&format=pjpg&auto=webp&s=7a611e0a1ea93eb732df65c0804bdc22ad067be9\n\nTrying NSFW RP aspects of Meta-Llama-3-8B-Instruct-GGUF and saw these responses.'

In [164]:
# # Replace 'column_name' with the column you want to check
# rows_with_none = df_unique[df_unique['snippet'].isna()]

# # Display the result
# print(len(rows_with_none))


In [137]:
def is_code(text):
    # Keywords commonly found in code
    code_keywords = ["def ", "class ", "import ", "console.log", "function", "return"]
    
    # Count symbols commonly found in code
    symbols = ["{", "}", "=", "()", "[]", ";", "<", ">"]
    symbol_count = sum(text.count(sym) for sym in symbols)
    
    # Heuristic: if any keyword is present or too many symbols, consider it code
    if any(keyword in text for keyword in code_keywords) or symbol_count > 10:
        return True
    return False


df_unique['is_code'] = df_unique['snippet'].apply(is_code)

KeyError: 'snippet'

In [165]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Make sure you have the required resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

# Custom stopwords: keep negations
stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "never"}

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    # Lowercasing
    text = text.lower()
    
    # Remove inline code snippets (between backticks or <code> tags)
    text = re.sub(r"`[^`]+`", " <CODE> ", text)
    text = re.sub(r"<code>.*?</code>", " <CODE> ", text, flags=re.DOTALL)
    
    # Remove URLs, emails, and file paths
    text = re.sub(r"http\S+|www\S+", " <URL> ", text)
    text = re.sub(r"\S+@\S+", " <EMAIL> ", text)
    text = re.sub(r"(/[A-Za-z0-9_\-\.]+)+", " <PATH> ", text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove non-alphabetic tokens, filter stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return " ".join(tokens)


df_unique["snippet"] = df_unique["reviews"].apply(preprocess_text)
print(df)


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ardacanseradali/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ardacanseradali/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ardacanseradali/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                                model_id  \
0              LoneStriker/Meta-Llama-3-8B-Instruct-GGUF   
1      SandLogicTechnologies/Meta-Llama-3-8B-Instruct...   
2                AI-Engine/Meta-Llama-3-8B-Instruct-GGUF   
3                 PawanKrd/Meta-Llama-3-8B-Instruct-GGUF   
4            MaziyarPanahi/Meta-Llama-3-8B-Instruct-GGUF   
...                                                  ...   
21759                      bartowski/gemma-2-27b-it-GGUF   
21760                  microsoft/trocr-small-handwritten   
21761      jonatasgrosman/wav2vec2-large-xlsr-53-spanish   
21762      jonatasgrosman/wav2vec2-large-xlsr-53-spanish   
21763                             google/madlad400-3b-mt   

                                topic  source  score  \
0       Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
1       Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
2       Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   
3       Meta-Llama-3-8B-Instruct-GGUF  reddit    6.0   

In [166]:
df_unique.to_csv("llm_check_reviews/mentioned_reviews_unique_preprocessed.csv", index=False, encoding="utf-8")


In [167]:
# Calculate the length of each review
df_unique['char_count'] = df_unique['reviews'].apply(len)

# Get the top 10 reviews with the most characters
top10 = df_unique.sort_values(by='char_count', ascending=False).head(3000)

print(top10[['reviews', 'char_count']])

                                                 reviews  char_count
13202  Some weights of the model checkpoint at TheBlo...       46726
4428   Some weights of the model checkpoint at TheBlo...       45881
12896  model = AutoModelForCausalLM.from_pretrained(\...       35157
9530   [https://www.reddit.com/r/StableDiffusion/comm...       32659
12098  2024-08-29 10:40:45.296907: W tensorflow/compi...       32373
...                                                  ...         ...
6495   ```\naccelerate launch main.py \\n        --mo...         481
11595  Hi,\r\n\r\nI need help with this error and hav...         481
10910  I found that the json configs at the time this...         481
8077   ```\nimport torch\nfrom h2oai_pipeline import ...         480
9681   **WizardLM 13b** (this is lora trained, so if ...         480

[3000 rows x 2 columns]


In [168]:
import spacy
nlp = spacy.load("en_core_web_sm")


In [169]:
import nltk
nltk.download("punkt")


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ardacanseradali/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [127]:
from nltk.tokenize import sent_tokenize

def extract_name_snippet(text, name, max_chars=512):
    """
    Returns a snippet of text (~max_chars) containing the specified name.
    If the sentence is longer than max_chars, centers snippet around the name.
    """
    sentences = sent_tokenize(text)
    
    for idx, sentence in enumerate(sentences):
        if name.lower() in sentence.lower():
            if len(sentence) > max_chars:
                # Find where the name appears
                pos = sentence.lower().find(name.lower())
                
                # Center the window around the name
                start = max(0, pos - max_chars // 2)
                end = min(len(sentence), start + max_chars)
                
                snippet = sentence[start:end].strip()
            else:
                # Normal case: add sentence and following ones if space allows
                snippet = sentence + " "
                next_idx = idx + 1
                while len(snippet) < max_chars and next_idx < len(sentences):
                    snippet += sentences[next_idx] + " "
                    next_idx += 1
                snippet = snippet.strip()
            
            return snippet  # stop after first relevant snippet
    
    return None



# Example u
text = df_unique["reviews"].iloc[5]
name = df_unique["topic"].iloc[5]
snippet = extract_name_snippet(text, name)
print(len(text))
print(snippet)
print("Length:", len(snippet))


141
If it's Llama.cpp, you should follow the template correctly: https://huggingface.co/MaziyarPanahi/Meta-Llama-3-8B-Instruct-GGUF/discussions/5
Length: 141


In [27]:
# from nltk.tokenize import sent_tokenize

# def extract_name_snippet(text, name, max_chars=512):
#     """
#     Returns a snippet of text (~max_chars) containing the specified name.
#     If the sentence is longer than max_chars, centers snippet around the name.
#     """
#     doc = nlp(text)
#     sentences = [sent.text for sent in doc.sents]
    
#     for idx, sentence in enumerate(sentences):
#         if name.lower() in sentence.lower():
#             if len(sentence) > max_chars:
#                 # Find where the name appears
#                 pos = sentence.lower().find(name.lower())
                
#                 # Center the window around the name
#                 start = max(0, pos - max_chars // 2)
#                 end = min(len(sentence), start + max_chars)
                
#                 snippet = sentence[start:end].strip()
#             else:
#                 # Normal case: add sentence and following ones if space allows
#                 snippet = sentence + " "
#                 next_idx = idx + 1
#                 while len(snippet) < max_chars and next_idx < len(sentences):
#                     snippet += sentences[next_idx] + " "
#                     next_idx += 1
#                 snippet = snippet.strip()
            
#             return snippet  # stop after first relevant snippet
    
#     return None



# # Example u
# text = df_unique["reviews"].iloc[5]
# name = df_unique["topic"].iloc[5]
# snippet = extract_name_snippet(text, name)
# print(len(text))
# print(snippet)
# print("Length:", len(snippet))


141
If it's Llama.cpp, you should follow the template correctly: https://huggingface.co/MaziyarPanahi/Meta-Llama-3-8B-Instruct-GGUF/discussions/5
Length: 141


In [ ]:
# import spacy

# # Load SpaCy model
# nlp = spacy.load("en_core_web_sm")

# # Compute number of sentences per review using SpaCy
# def count_sentences(text):
#     doc = nlp(str(text))
#     sentences = [sent.text for sent in doc.sents]
#     return len(sentences)

# sentence_counts = df['reviews'].apply(count_sentences)

# # Get top 10 indices
# top10_idx = sentence_counts.nlargest(10).index

# # Print review and sentence count together
# for idx in top10_idx:
#     doc = nlp(str(df.loc[idx, 'reviews']))
#     sentences = [sent.text for sent in doc.sents]
#     print(f"\n--- Review index: {idx} | Sentence count: {sentence_counts[idx]} ---")




--- Review index: 10062 | Sentence count: 214 ---

--- Review index: 2737 | Sentence count: 201 ---

--- Review index: 2738 | Sentence count: 201 ---

--- Review index: 147 | Sentence count: 196 ---

--- Review index: 1571 | Sentence count: 196 ---

--- Review index: 1583 | Sentence count: 196 ---

--- Review index: 1602 | Sentence count: 196 ---

--- Review index: 1681 | Sentence count: 196 ---

--- Review index: 1737 | Sentence count: 196 ---

--- Review index: 1793 | Sentence count: 196 ---


In [ ]:
# # Get top 10 indices
# top10_idx = sentence_counts.nlargest(100).index

# # Print review and sentence count together
# for idx in top10_idx:
#     doc = nlp(str(df.loc[idx, 'reviews']))
#     sentences = [sent.text for sent in doc.sents]
#     print(f"\n--- Review index: {idx} | Sentence count: {sentence_counts[idx]} ---")


--- Review index: 10062 | Sentence count: 214 ---

--- Review index: 2737 | Sentence count: 201 ---

--- Review index: 2738 | Sentence count: 201 ---

--- Review index: 147 | Sentence count: 196 ---

--- Review index: 1571 | Sentence count: 196 ---

--- Review index: 1583 | Sentence count: 196 ---

--- Review index: 1602 | Sentence count: 196 ---

--- Review index: 1681 | Sentence count: 196 ---

--- Review index: 1737 | Sentence count: 196 ---

--- Review index: 1793 | Sentence count: 196 ---

--- Review index: 1850 | Sentence count: 196 ---

--- Review index: 2640 | Sentence count: 196 ---

--- Review index: 3765 | Sentence count: 196 ---

--- Review index: 5312 | Sentence count: 196 ---

--- Review index: 5504 | Sentence count: 196 ---

--- Review index: 5514 | Sentence count: 196 ---

--- Review index: 6672 | Sentence count: 196 ---

--- Review index: 7864 | Sentence count: 196 ---

--- Review index: 8439 | Sentence count: 196 ---

--- Review index: 9090 | Sentence count: 196 ---


KeyboardInterrupt: 

In [128]:
# First add a snippet column
df_unique['snippet'] = df_unique.apply(
    lambda row: extract_name_snippet(str(row['snippet']), row['topic']),
    axis=1
)

# 7. Get Score

In [ ]:
# def sentiment_score(review):
#     tokens = tokenizer.encode(review, return_tensors='pt')
    
#     result = model(tokens)
#     return int(torch.argmax(result.logits))+1

In [170]:
def sentiment_score(review):
    inputs = tokenizer(
        review,
        return_tensors='pt',
        truncation=True,   # <-- cuts off after 512 tokens
        max_length=512,
        padding='max_length'
    )
    with torch.no_grad():
        result = model(**inputs)
    return int(torch.argmax(result.logits)) + 1


In [171]:
sentiment_score(df_unique['snippet'].iloc[1])

3

In [172]:

# Then compute sentiment on the snippet (if it exists)
df_unique['sentiment'] = df_unique['snippet'].apply(
    lambda x: sentiment_score(x) if x else 0
)

In [ ]:
# df_unique['sentiment'] = df_unique['reviews'].apply(lambda x: sentiment_score(x[:512]))


In [173]:
# Count all sentiment values
sent_counts = df_unique['sentiment'].value_counts()

# Extract counts for 1, 2, 3, 4, 5
count_0 = sent_counts.get(0, 0)
count_1 = sent_counts.get(1, 0)
count_2 = sent_counts.get(2, 0)
count_3 = sent_counts.get(3, 0)
count_4 = sent_counts.get(4, 0)
count_5 = sent_counts.get(5, 0)

print(f"Number of 0-star sentiments: {count_0}")
print(f"Number of 1-star sentiments: {count_1}")
print(f"Number of 2-star sentiments: {count_2}")
print(f"Number of 3-star sentiments: {count_3}")
print(f"Number of 4-star sentiments: {count_4}")
print(f"Number of 5-star sentiments: {count_5}")

print("=================================")

total = df_unique['sentiment'].notnull().sum()  # ignores None/NaN
print(f"Percentage of 1-star: {count_1/total*100:.2f}%")
print(f"Percentage of 2-star: {count_2/total*100:.2f}%")
print(f"Percentage of 3-star: {count_3/total*100:.2f}%")
print(f"Percentage of 4-star: {count_4/total*100:.2f}%")
print(f"Percentage of 5-star: {count_5/total*100:.2f}%")


Number of 0-star sentiments: 124
Number of 1-star sentiments: 5639
Number of 2-star sentiments: 490
Number of 3-star sentiments: 2927
Number of 4-star sentiments: 2885
Number of 5-star sentiments: 2567
Percentage of 1-star: 38.54%
Percentage of 2-star: 3.35%
Percentage of 3-star: 20.00%
Percentage of 4-star: 19.72%
Percentage of 5-star: 17.54%


In [174]:
df_unique.to_csv("llm_check_reviews/mentioned_reviews_sentiment_preprocessed.csv", index=False, encoding="utf-8")

In [109]:
df_filtered = df_unique[df_unique['is_code'] == False]

KeyError: 'is_code'

In [ ]:
print(len(df_filtered))
print(len(df_unique))

1671
7269


In [ ]:
df_filtered.to_csv("llm_check_reviews/mentioned_reviews_sentiment_filtered.csv", index=False, encoding="utf-8")